In [1]:
import os
import pandas as pd
import pickle # Changed from joblib to pickle
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import SVC

# 1. Create model directory if it doesn't exist
if not os.path.exists('model'):
    os.makedirs('model')

# 2. Load Dataset (Breast Cancer: 30 features, 569 instances)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

# Combine to save the test set later
df = pd.concat([X, y], axis=1)

# Split the data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Save the test data for Streamlit upload (contains features + target)
test_data = pd.concat([X_test, y_test], axis=1)
test_data.to_csv('test_data.csv', index=False)
print("Saved test_data.csv")

# 3. Initialize Models
models = {
    'Logistic Regression': LogisticRegression(max_iter=10000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB(),
    'Support Vector Machine': CalibratedClassifierCV(SVC(random_state=42), ensemble=False)
}

# 4. Train and Save Models with Protocol 4
for name, model in models.items():
    model.fit(X_train, y_train)
    filename = f"model/{name.replace(' ', '_').lower()}.pkl"
    
    # Save using standard pickle with protocol 4
    with open(filename, 'wb') as f:
        pickle.dump(model, f, protocol=4)
        
    print(f"Trained and saved: {name}")

print("\nAll models trained and saved successfully! You can now run the Streamlit app.")

Saved test_data.csv
Trained and saved: Logistic Regression
Trained and saved: Decision Tree
Trained and saved: KNN
Trained and saved: Naive Bayes
Trained and saved: Support Vector Machine

All models trained and saved successfully! You can now run the Streamlit app.


In [2]:
%%writefile app.py
import streamlit as st
import pandas as pd
import pickle # Changed from joblib to pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef, confusion_matrix, classification_report

st.set_page_config(page_title="ML Model Evaluator", layout="wide")

st.title("Machine Learning Classification Evaluation App")
st.markdown("Upload your `test_data.csv`, select a trained model, and view the evaluation metrics.")

uploaded_file = st.file_uploader("Upload Test Dataset (CSV format)", type=["csv"])

model_options = {
    'Logistic Regression': 'model/logistic_regression.pkl',
    'Decision Tree': 'model/decision_tree.pkl',
    'KNN': 'model/knn.pkl',
    'Naive Bayes': 'model/naive_bayes.pkl',
    'Support Vector Machine': 'model/support_vector_machine.pkl'
}

selected_model_name = st.selectbox("Select Classification Model", list(model_options.keys()))

if uploaded_file is not None:
    df_test = pd.read_csv(uploaded_file)
    st.write("### Data Preview")
    st.dataframe(df_test.head())
    
    if 'target' not in df_test.columns:
        st.error("The uploaded CSV must contain a 'target' column.")
    else:
        X_test = df_test.drop(columns=['target'])
        y_test = df_test['target']
        model_path = model_options[selected_model_name]
        
        try:
            # Load model using standard pickle
            with open(model_path, 'rb') as f:
                model = pickle.load(f)
                
            y_pred = model.predict(X_test)
            y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
            
            acc = accuracy_score(y_test, y_pred)
            prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
            rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
            f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
            mcc = matthews_corrcoef(y_test, y_pred)
            auc = roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else "N/A"
            
            st.subheader(f"Evaluation Metrics for {selected_model_name}")
            col1, col2, col3, col4, col5, col6 = st.columns(6)
            col1.metric("Accuracy", f"{acc:.4f}")
            col2.metric("AUC Score", f"{auc:.4f}" if isinstance(auc, float) else auc)
            col3.metric("Precision", f"{prec:.4f}")
            col4.metric("Recall", f"{rec:.4f}")
            col5.metric("F1 Score", f"{f1:.4f}")
            col6.metric("MCC Score", f"{mcc:.4f}")
            
            st.write("---")
            col_cm, col_cr = st.columns(2)
            
            with col_cm:
                st.subheader("Confusion Matrix")
                cm = confusion_matrix(y_test, y_pred)
                fig, ax = plt.subplots(figsize=(4, 3))
                sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
                ax.set_xlabel("Predicted")
                ax.set_ylabel("Actual")
                st.pyplot(fig)
                
            with col_cr:
                st.subheader("Classification Report")
                report = classification_report(y_test, y_pred, output_dict=True)
                df_report = pd.DataFrame(report).transpose()
                st.dataframe(df_report.style.background_gradient(cmap='Blues'))

        except FileNotFoundError:
            st.error(f"Model file not found at `{model_path}`.")
else:
    st.info("Please upload the test data CSV to begin evaluation.")

Overwriting app.py


In [3]:
!python -m streamlit run app.py

^C


In [5]:
import sys
!{sys.executable} -m pip install streamlit

     ---------------------------------------- 0.0/62.1 kB ? eta -:--:--
     ---------------------------------------- 62.1/62.1 kB 3.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
   - -------------------------------------- 0.4/11.2 MB 8.1 MB/s eta 0:00:02
   -- ------------------------------------- 0.8/11.2 MB 8.7 MB/s eta 0:00:02
   ---- ----------------------------------- 1.3/11.2 MB 9.3 MB/s eta 0:00:02
   ------ --------------------------------- 1.8/11.2 MB 8.6 MB/s eta 0:00:02
   ------- -------------------------------- 2.2/11.2 MB 8.9 MB/s eta 0:00:02
   --------- ------------------------------ 2.7/11.2 MB 8.9 MB/s eta 0:00:01
   ----------- ---------------------------- 3.1/11.2 MB 9.1 MB/s eta 0:00:01
   ------------- -------------------------- 3.7/11.2 MB 9.3 MB/s eta 0:00:01
   -------------- ------------------------- 4.2/11.2 MB 9.5 MB/s eta 0:00:01
   ---------------- ----------------------- 4.7/11.2 MB 9.5 MB/s eta 0:00:01
   -----


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: C:\Users\Anjali\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [ ]:
import sys
!{sys.executable} -m streamlit run app.py

In [8]:
import sys
!{sys.executable} -m pip install --upgrade numpy

^C
